In [1]:
import pandas as pd

ratings = pd.read_csv('ratings.csv')
movies  = pd.read_csv('movies.csv')
tags    = pd.read_csv('tags.csv')
links   = pd.read_csv('links.csv')

print(ratings.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [2]:
for name, df in [('ratings', ratings), ('movies', movies), ('tags', tags), ('links', links)]:
    print(f"--- {name} ---")
    print(df.shape)
    print(df.isnull().sum())
    print("Duplicates:", df.duplicated().sum())

--- ratings ---
(100836, 4)
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Duplicates: 0
--- movies ---
(9742, 3)
movieId    0
title      0
genres     0
dtype: int64
Duplicates: 0
--- tags ---
(3683, 4)
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
Duplicates: 0
--- links ---
(9742, 3)
movieId    0
imdbId     0
tmdbId     8
dtype: int64
Duplicates: 0


The MovieLens ml-latest-small dataset was loaded and all four tables (ratings, movies, tags, and links) were inspected for missing values, duplicate rows, and inconsistencies.
The ratings table contains 100,836 rows and 4 columns, the movies table contains 9,742 rows and 3 columns, the tags table contains 3,683 rows and 4 columns, and the links table contains 9,742 rows and 3 columns.
#No missing values were found in the ratings, movies, or tags tables.
# The links table has 8 missing values in the tmdbId column, which were left unchanged because this table is not used in building the recommendation model.
#No duplicate rows were found in any of the four tables, so no rows were removed.

In [3]:
# Rating values should be 0.5 to 5 in half steps
print(sorted(ratings.rating.unique()))

# Every rated movie should exist in the movies table
print("Orphan ratings:", (~ratings.movieId.isin(movies.movieId)).sum())

# Movies with no genres listed
print("No genres:", (movies.genres == '(no genres listed)').sum())

# How sparse is the data?
print(ratings.groupby('userId').size().describe())
print(ratings.groupby('movieId').size().describe())

[np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
Orphan ratings: 0
No genres: 34
count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64
count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
dtype: float64


In [4]:
#Handle the 34 movies with no genres
movies['genres'] = movies['genres'].replace('(no genres listed)', 'Unknown')

34 movies with no genre relabelled unknown
to keep the data intact.
just incase of hybrid approach.

In [5]:
#filter rarely_rated movies
counts = ratings.movieId.value_counts()
ratings_clean = ratings[ratings.movieId.isin(counts[counts >= 5].index)]
print("Before:", ratings.shape[0], "After:", ratings_clean.shape[0])
print("Movies before:", ratings.movieId.nunique(), "After:", ratings_clean.movieId.nunique())

Before: 100836 After: 90274
Movies before: 9724 After: 3650


Movies with fewer than 5 ratings were removed to reduce sparsity. This reduced the ratings from 100,836 to 90,274 (about 10.5% removed) and the number of movies from 9,724 to 3,650 (about 62.5% removed). Most of the removed movies had very few ratings, so most of the rating data was kept.

In [6]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings_clean, test_size=0.2, random_state=42)
print(train.shape, test.shape)

ratings_clean.to_csv('ratings_clean.csv', index=False)
movies.to_csv('movies_clean.csv', index=False)
train.to_csv('train.csv', index=False)
test.to_csv('test.csv', index=False)

(72219, 4) (18055, 4)


The MovieLens ml-latest-small dataset was loaded and all four tables (ratings, movies, tags, and links) were inspected for missing values, duplicates, and inconsistencies. The ratings table contains 100,836 rows, the movies table 9,742 rows, the tags table 3,683 rows, and the links table 9,742 rows.
- No missing values or duplicate rows were found in ratings, movies, or tags.
- The links table has 8 missing tmdbId values, left unchanged because this table is not used in modelling.
- Rating values were verified to lie between 0.5 and 5.0 in half-point steps, and no ratings referenced movies missing from the movies table. - Movies(34) had no genre listed; these were relabelled “Unknown” rather than removed, to keep the data intact in case a hybrid approach using genres is adopted.
- Each of the 610 users had at least 20 ratings, so no users were removed.
- The number of ratings per movie was highly skewed, so movies with fewer than 5 ratings were removed to reduce sparsity. This reduced the ratings from 100,836 to 90,274 (about 10.5% removed) and the number of movies from 9,724 to 3,650 (about 62.5% removed), meaning most of the rating data was retained. The movies table was kept in full (9,742 movies) so that any movie can be looked up by title and genre, even though only the 3,650 movies with at least 5 ratings remain in the cleaned ratings data.
- The relevant columns for collaborative filtering are userId, movieId, and rating, with title and genres kept for presenting recommendations.
- The cleaned ratings were split 80/20 into a training set (72,219 rows) and a test set (18,055 rows) using a fixed random seed (42) for reproducibility.